# Table 5 Evaluation: removed-part-aware vs largest-hole-only (scan all 100 samples)

This notebook is designed to be placed under `evaluation/` and run directly.

It compares two baselines under the same planar repair geometry:

- `planar_removed_part_aware`
- `planar_largest_hole_only`

Main behavior:

1. Automatically scan the dataset root and collect all valid sample folders (target: 100 samples).
2. Run the two baselines on every valid sample.
3. Save detailed per-sample results and a Table-5-style summary.
4. Save failure / disagreement cases for inspection.


In [7]:
from pathlib import Path
import pandas as pd
import numpy as np
import traceback
import math
import sys
pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 200)


In [8]:
# ===== Paths =====
# Adjust this if needed. The notebook assumes it is placed in evaluation/.

NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
# Candidate dataset roots. The first existing one will be used.
DATASET_CANDIDATES = [
    PROJECT_ROOT / 'dataset',
    PROJECT_ROOT / 'data' / 'dataset',
    PROJECT_ROOT / 'data',
]

DATASET_ROOT = None
for p in DATASET_CANDIDATES:
    if p.exists():
        DATASET_ROOT = p
        break

if DATASET_ROOT is None:
    raise FileNotFoundError(
        'Could not find dataset root. Please edit DATASET_CANDIDATES in this notebook.'
    )

RESULT_DIR = PROJECT_ROOT / 'results' / 'csv'
RESULT_DIR.mkdir(parents=True, exist_ok=True)

DETAIL_CSV = RESULT_DIR / 'table5_detail_all100.csv'
SUMMARY_CSV = RESULT_DIR / 'table5_summary.csv'
FAILURE_CSV = RESULT_DIR / 'table5_failure_cases.csv'
DISAGREE_CSV = RESULT_DIR / 'table5_disagreement_cases.csv'

print('PROJECT_ROOT =', PROJECT_ROOT)
print('DATASET_ROOT =', DATASET_ROOT)
print('RESULT_DIR   =', RESULT_DIR)


PROJECT_ROOT = D:\MyJupyter\Works\3Dsegment
DATASET_ROOT = D:\MyJupyter\Works\3Dsegment\dataset
RESULT_DIR   = D:\MyJupyter\Works\3Dsegment\results\csv


In [9]:
# ===== Imports =====
# Change these import lines only if your baseline module paths/names differ.
from baselines.baseline_planar_triangulation import repair_one_sample_planar
from baselines.baseline_planar_largest_hole import repair_one_sample_planar_largest_hole

BASELINES = {
    'planar_removed_part_aware': repair_one_sample_planar,
    'planar_largest_hole_only': repair_one_sample_planar_largest_hole,
}

print('Loaded baselines:', list(BASELINES.keys()))


Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.
Loaded baselines: ['planar_removed_part_aware', 'planar_largest_hole_only']


In [10]:
# ===== Sample discovery =====
REQUIRED_FILES = ['M_gt.obj', 'M_d.obj']
OPTIONAL_FILES = ['meta.json']

def is_valid_sample_dir(sample_dir: Path) -> bool:
    if not sample_dir.is_dir():
        return False
    return all((sample_dir / fn).exists() for fn in REQUIRED_FILES)

def discover_sample_dirs(dataset_root: Path):
    dirs = []
    for p in sorted(dataset_root.iterdir(), key=lambda x: x.name):
        if is_valid_sample_dir(p):
            dirs.append(p)
    return dirs

sample_dirs = discover_sample_dirs(DATASET_ROOT)
sample_ids = [p.name for p in sample_dirs]

print(f'Found {len(sample_dirs)} valid sample dirs')
if len(sample_dirs) != 100:
    print('Warning: expected about 100 samples, but found', len(sample_dirs))

sample_ids[:10]


Found 100 valid sample dirs


['2101',
 '2168',
 '2210',
 '2269',
 '2398',
 '2549',
 '2594',
 '2634',
 '2689',
 '3022']

In [11]:
# ===== Evaluation config =====
# Success threshold based on residual target perimeter after repair.
# Adjust if your paper uses a different threshold.
SUCCESS_THRESHOLD = 1e-6

# Metrics we try to fetch from baseline output dict.
METRIC_KEYS = [
    'total_loop_len_before',
    'nearest_loop_len_after',
    'improvement',
    'num_new_vertices',
    'num_added_faces',
    'mean_added_face_quality',
    'min_added_face_quality',
    'num_added_faces_inside_zone',
    'num_added_faces_outside_zone',
    'face_locality_ratio',
    'max_boundary_loop_before',
    'max_boundary_loop_after',
]

def to_float_or_nan(x):
    try:
        if x is None:
            return np.nan
        return float(x)
    except Exception:
        return np.nan

def infer_success(row_dict):
    # Priority 1: explicit success from baseline output
    if 'success' in row_dict and row_dict['success'] is not None:
        return bool(row_dict['success'])

    # Priority 2: infer from nearest_loop_len_after
    after = to_float_or_nan(row_dict.get('nearest_loop_len_after'))
    if not np.isnan(after):
        return after <= SUCCESS_THRESHOLD

    return False


In [12]:
# ===== Core runner =====
def evaluate_one(sample_dir: Path, baseline_name: str, fn):
    row = {
        'sample_id': sample_dir.name,
        'sample_dir': str(sample_dir),
        'baseline': baseline_name,
        'ran_ok': False,
        'success': False,
        'error_msg': '',
        'traceback': '',
    }

    try:
        out = fn(sample_dir)

        if not isinstance(out, dict):
            row['error_msg'] = f'baseline returned non-dict: {type(out)}'
            return row

        row['ran_ok'] = True

        for k in METRIC_KEYS:
            row[k] = out.get(k, np.nan)

        if 'nearby_loops' in out:
            try:
                row['num_target_loops'] = len(out['nearby_loops'])
            except Exception:
                row['num_target_loops'] = np.nan

        if 'selected_loop_index' in out:
            row['selected_loop_index'] = out.get('selected_loop_index', np.nan)

        if 'selected_loop_len_before' in out:
            row['selected_loop_len_before'] = out.get('selected_loop_len_before', np.nan)

        row['success'] = infer_success({**row, **out})
        return row

    except Exception as e:
        row['error_msg'] = f'{type(e).__name__}: {e}'
        row['traceback'] = traceback.format_exc()
        return row

rows = []
total_jobs = len(sample_dirs) * len(BASELINES)
job_idx = 0

for sample_dir in sample_dirs:
    for baseline_name, fn in BASELINES.items():
        job_idx += 1
        print(f'[{job_idx}/{total_jobs}] sample={sample_dir.name}, baseline={baseline_name}')
        rows.append(evaluate_one(sample_dir, baseline_name, fn))

df = pd.DataFrame(rows)
df.to_csv(DETAIL_CSV, index=False, encoding='utf-8-sig')
print('Saved detail csv to', DETAIL_CSV)
print(df.shape)
df.head()


[1/200] sample=2101, baseline=planar_removed_part_aware
[2/200] sample=2101, baseline=planar_largest_hole_only
[3/200] sample=2168, baseline=planar_removed_part_aware
[4/200] sample=2168, baseline=planar_largest_hole_only
[5/200] sample=2210, baseline=planar_removed_part_aware
[6/200] sample=2210, baseline=planar_largest_hole_only
[7/200] sample=2269, baseline=planar_removed_part_aware
[8/200] sample=2269, baseline=planar_largest_hole_only
[9/200] sample=2398, baseline=planar_removed_part_aware
[10/200] sample=2398, baseline=planar_largest_hole_only
[11/200] sample=2549, baseline=planar_removed_part_aware
[12/200] sample=2549, baseline=planar_largest_hole_only
[13/200] sample=2594, baseline=planar_removed_part_aware
[14/200] sample=2594, baseline=planar_largest_hole_only
[15/200] sample=2634, baseline=planar_removed_part_aware
[16/200] sample=2634, baseline=planar_largest_hole_only
[17/200] sample=2689, baseline=planar_removed_part_aware
[18/200] sample=2689, baseline=planar_largest_ho

,sample_id,sample_dir,baseline,ran_ok,success,error_msg,traceback,total_loop_len_before,nearest_loop_len_after,improvement,num_new_vertices,num_added_faces,mean_added_face_quality,min_added_face_quality,num_added_faces_inside_zone,num_added_faces_outside_zone,face_locality_ratio,max_boundary_loop_before,max_boundary_loop_after,num_target_loops
0,2101,D:\MyJupyter\Works\3Dsegment\dataset\2101,planar_removed_part_aware,True,True,,,1.681047,0.482893,1.198153,0,108.0,0.616440,0.352994,103.0,5.0,0.953704,NaN,NaN,5.0
1,2101,D:\MyJupyter\Works\3Dsegment\dataset\2101,planar_largest_hole_only,True,True,,,0.521767,1.642172,-1.120405,0,34.0,0.680795,0.409968,34.0,0.0,1.000000,NaN,NaN,NaN
2,2168,D:\MyJupyter\Works\3Dsegment\dataset\2168,planar_removed_part_aware,True,True,,,0.159273,0.000000,0.159273,0,128.0,0.282756,0.059004,0.0,128.0,0.000000,NaN,NaN,1.0
3,2168,D:\MyJupyter\Works\3Dsegment\dataset\2168,planar_largest_hole_only,True,True,,,0.348841,0.000000,0.348841,0,47.0,0.358772,0.094426,0.0,47.0,0.000000,NaN,NaN,NaN
4,2210,D:\MyJupyter\Works\3Dsegment\dataset\2210,planar_removed_part_aware,True,True,,,4.811912,0.000000,4.811912,0,186.0,0.230950,0.043407,0.0,186.0,0.000000,NaN,NaN,1.0


In [13]:
# ===== Summary for Table 5 =====
def safe_mean(series):
    s = pd.to_numeric(series, errors='coerce')
    if s.notna().sum() == 0:
        return np.nan
    return s.mean()

def safe_median(series):
    s = pd.to_numeric(series, errors='coerce')
    if s.notna().sum() == 0:
        return np.nan
    return s.median()

summary_rows = []
for baseline_name, g in df.groupby('baseline'):
    row = {
        'baseline': baseline_name,
        'n_total': len(g),
        'n_ran_ok': int(g['ran_ok'].fillna(False).sum()),
        'n_success': int(g['success'].fillna(False).sum()),
        'success_rate': float(g['success'].fillna(False).mean()),
        'nearest_loop_len_after_mean': safe_mean(g['nearest_loop_len_after']),
        'nearest_loop_len_after_median': safe_median(g['nearest_loop_len_after']),
        'improvement_mean': safe_mean(g['improvement']),
        'num_added_faces_mean': safe_mean(g['num_added_faces']),
        'num_added_faces_outside_zone_mean': safe_mean(g['num_added_faces_outside_zone']),
        'face_locality_ratio_mean': safe_mean(g['face_locality_ratio']),
        'mean_added_face_quality_mean': safe_mean(g['mean_added_face_quality']),
    }
    summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows).sort_values('baseline').reset_index(drop=True)
summary_df.to_csv(SUMMARY_CSV, index=False, encoding='utf-8-sig')
print('Saved summary csv to', SUMMARY_CSV)
summary_df


Saved summary csv to D:\MyJupyter\Works\3Dsegment\results\csv\table5_summary.csv


,baseline,n_total,n_ran_ok,n_success,success_rate,nearest_loop_len_after_mean,nearest_loop_len_after_median,improvement_mean,num_added_faces_mean,num_added_faces_outside_zone_mean,face_locality_ratio_mean,mean_added_face_quality_mean
0,planar_largest_hole_only,100,100,100,1.0,0.797338,0.375953,0.812341,78.202247,74.292135,0.081214,0.401867
1,planar_removed_part_aware,100,100,100,1.0,0.169078,0.000000,0.990081,77.370787,42.303371,0.724519,0.483989


In [14]:
# ===== Failure cases =====
failure_df = df[(df['ran_ok'] == False) | (df['success'] == False)].copy()
failure_df = failure_df.sort_values(['baseline', 'sample_id']).reset_index(drop=True)
failure_df.to_csv(FAILURE_CSV, index=False, encoding='utf-8-sig')
print('Saved failure csv to', FAILURE_CSV)
failure_df.head(20)


Saved failure csv to D:\MyJupyter\Works\3Dsegment\results\csv\table5_failure_cases.csv


,sample_id,sample_dir,baseline,ran_ok,success,error_msg,traceback,total_loop_len_before,nearest_loop_len_after,improvement,num_new_vertices,num_added_faces,mean_added_face_quality,min_added_face_quality,num_added_faces_inside_zone,num_added_faces_outside_zone,face_locality_ratio,max_boundary_loop_before,max_boundary_loop_after,num_target_loops


In [15]:
# ===== Pairwise comparison between the two baselines =====
pivot_cols = [
    'ran_ok', 'success', 'nearest_loop_len_after', 'improvement',
    'num_added_faces_outside_zone', 'face_locality_ratio',
]

wide = df.pivot(index='sample_id', columns='baseline', values=pivot_cols)
wide.columns = [f'{metric}__{baseline}' for metric, baseline in wide.columns]
wide = wide.reset_index()

aware = 'planar_removed_part_aware'
naive = 'planar_largest_hole_only'

for col in [
    f'success__{aware}', f'success__{naive}',
    f'nearest_loop_len_after__{aware}', f'nearest_loop_len_after__{naive}',
    f'num_added_faces_outside_zone__{aware}', f'num_added_faces_outside_zone__{naive}',
    f'face_locality_ratio__{aware}', f'face_locality_ratio__{naive}',
]:
    if col not in wide.columns:
        wide[col] = np.nan

wide['aware_win_by_success'] = (
    wide[f'success__{aware}'].fillna(False).astype(bool) &
    ~wide[f'success__{naive}'].fillna(False).astype(bool)
)

wide['naive_win_by_success'] = (
    wide[f'success__{naive}'].fillna(False).astype(bool) &
    ~wide[f'success__{aware}'].fillna(False).astype(bool)
)

wide['delta_nearest_loop_len_after'] = (
    pd.to_numeric(wide[f'nearest_loop_len_after__{naive}'], errors='coerce') -
    pd.to_numeric(wide[f'nearest_loop_len_after__{aware}'], errors='coerce')
)

wide['delta_outside_zone_faces'] = (
    pd.to_numeric(wide[f'num_added_faces_outside_zone__{naive}'], errors='coerce') -
    pd.to_numeric(wide[f'num_added_faces_outside_zone__{aware}'], errors='coerce')
)

wide['delta_face_locality_ratio'] = (
    pd.to_numeric(wide[f'face_locality_ratio__{aware}'], errors='coerce') -
    pd.to_numeric(wide[f'face_locality_ratio__{naive}'], errors='coerce')
)

disagree_df = wide[
    (wide['aware_win_by_success']) |
    (wide['naive_win_by_success']) |
    (wide['delta_nearest_loop_len_after'].abs() > 1e-12)
].copy()

disagree_df = disagree_df.sort_values(
    by=['aware_win_by_success', 'delta_nearest_loop_len_after'],
    ascending=[False, False]
).reset_index(drop=True)

disagree_df.to_csv(DISAGREE_CSV, index=False, encoding='utf-8-sig')
print('Saved disagreement csv to', DISAGREE_CSV)
disagree_df.head(20)


Saved disagreement csv to D:\MyJupyter\Works\3Dsegment\results\csv\table5_disagreement_cases.csv


C:\Users\Administrator\AppData\Local\Temp\ipykernel_21456\1383727126.py:24: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  wide[f'success__{aware}'].fillna(False).astype(bool) &
C:\Users\Administrator\AppData\Local\Temp\ipykernel_21456\1383727126.py:25: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  ~wide[f'success__{naive}'].fillna(False).astype(bool)
C:\Users\Administrator\AppData\Local\Temp\ipykernel_21456\1383727126.py:29: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call r

,sample_id,ran_ok__planar_largest_hole_only,ran_ok__planar_removed_part_aware,success__planar_largest_hole_only,success__planar_removed_part_aware,nearest_loop_len_after__planar_largest_hole_only,nearest_loop_len_after__planar_removed_part_aware,improvement__planar_largest_hole_only,improvement__planar_removed_part_aware,num_added_faces_outside_zone__planar_largest_hole_only,num_added_faces_outside_zone__planar_removed_part_aware,face_locality_ratio__planar_largest_hole_only,face_locality_ratio__planar_removed_part_aware,aware_win_by_success,naive_win_by_success,delta_nearest_loop_len_after,delta_outside_zone_faces,delta_face_locality_ratio
0,2689,True,True,True,True,7.23146,0.473734,-3.350723,6.757726,150.0,290.0,0.0,0.013605,False,False,6.757726,-140.0,0.013605
1,42350,True,True,True,True,6.312988,1.135537,-3.209172,7.920759,174.0,474.0,0.011364,0.066929,False,False,5.177450,-300.0,0.055565
2,37498,True,True,True,True,6.04406,1.319323,0.209711,10.382346,339.0,501.0,0.063536,0.25,False,False,4.724738,-162.0,0.186464
3,39546,True,True,True,True,4.775435,0.180901,-3.800666,4.594535,132.0,188.0,0.0,0.41433,False,False,4.594535,-56.0,0.414330
4,44015,True,True,True,True,6.522927,2.837498,-2.41898,7.789375,188.0,473.0,0.12963,0.162832,False,False,3.685429,-285.0,0.033202
5,40525,True,True,True,True,2.008916,0.0,-1.33023,2.008916,20.0,82.0,0.0,0.226415,False,False,2.008916,-62.0,0.226415
6,40114,True,True,True,True,1.841492,0.0,0.0,3.682984,93.0,186.0,0.05102,0.05102,False,False,1.841492,-93.0,0.000000
7,42103,True,True,True,True,1.476778,0.0,-0.641232,1.476778,26.0,0.0,0.0,1.0,False,False,1.476778,26.0,1.000000
8,35871,True,True,True,True,1.46136,0.0,3.364885,1.46136,78.0,0.0,0.0,1.0,False,False,1.461360,78.0,1.000000
9,39474,True,True,True,True,1.173107,0.0,-0.124332,1.173107,56.0,33.0,0.0,0.484375,False,False,1.173107,23.0,0.484375


In [16]:
# ===== Pretty table for paper writing =====
pretty = summary_df.copy()

def fmt(x, nd=4):
    if pd.isna(x):
        return 'NA'
    if isinstance(x, (int, np.integer)):
        return str(int(x))
    return f'{float(x):.{nd}f}'

display_cols = [
    'baseline', 'n_total', 'n_success', 'success_rate',
    'nearest_loop_len_after_mean', 'improvement_mean',
    'num_added_faces_outside_zone_mean', 'face_locality_ratio_mean',
]

pretty = pretty[display_cols].copy()
for col in pretty.columns:
    if col != 'baseline':
        pretty[col] = pretty[col].map(lambda v: fmt(v, 4))

pretty


,baseline,n_total,n_success,success_rate,nearest_loop_len_after_mean,improvement_mean,num_added_faces_outside_zone_mean,face_locality_ratio_mean
0,planar_largest_hole_only,100,100,1.0000,0.7973,0.8123,74.2921,0.0812
1,planar_removed_part_aware,100,100,1.0000,0.1691,0.9901,42.3034,0.7245


## Notes

- If your baseline functions require different import paths, edit the import cell only.
- If your success criterion is not `nearest_loop_len_after <= 1e-6`, edit `SUCCESS_THRESHOLD`.
- If you already have a detailed all-samples CSV from a previous run, you can also adapt this notebook to read that CSV instead of re-running the baselines.
